# Complexity-08e — Le compte à rebours : des machines finies qui attendent une contradiction (NF, C₀)

**Navigation** : [<< Complexity-08d — Indépendance prouvée](Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb) | [Version Python](Complexity-08e-NF-MachinesCompteARebours-Python.ipynb) | [Index de la série](README.md)
**Public : Recherche.** Le carnet ne suppose que les carnets précédents de la série : les machines de Turing pas à pas (01), la notion de réduction (02), le castor affairé et sa croissance (07b).

> **Statut épistémique.** Tout ce qui suit distingue trois natures de résultats : un **théorème Lean formalisé et audité**, un **calcul reproductible vérifié par simulateur indépendant**, et une **borne conditionnelle**. Les artefacts du dépôt `wlad-svennik/notes` sont **cités et épinglés par révision** ; aucun contenu n'est recopié dans ce dépôt (aucune licence détectée dans le dépôt source au 20/11/2026, demande de clarification ouverte : `wlad-svennik/notes#1`). La section de mécanique exécute **notre propre** machine-jouet, déclarée comme telle : elle reproduit le *câblage* du compte à rebours, jamais les machines réelles.

## 0. Trois statuts pour une même famille d'objets

Le dépôt source publie une famille de machines à nombre d'états décroissant (210, 201, 192, 185, 184) et, en parallèle, des machines calibrées sur une autre théorie (198 états pour C₀). Avant de citer un seul chiffre, la règle du carnet 08d s'applique ici mot pour mot : **classer l'objet avant de l'affirmer**.

1. **Théorème Lean formalisé et audité.** Pour `nf-bb185`, l'équivalence est un théorème non borné, `NFMachine185.halts_iff_nf_refutation`, accompagné d'un audit d'axiomes enregistré : trois axiomes standard de Lean et onze vérifications finies par évaluation native. Pour `nf-bb192`, l'équivalence a été vérifiée par Lean 4.35.0 sur la table littérale. C'est le statut le plus fort de la famille.
2. **Calcul reproductible.** Pour `nf-bb184`, le dépôt publie la table de la machine (empreinte SHA-256 annoncée `7764cc84…8798`), un simulateur indépendant (142 608 essais, 88 555 587 pas binaires), la reconstruction des 2 809 instructions émettrices d'axiomes, un point de contrôle de vérification de 1,84 Mo et un `SHA256SUMS`. Ce n'est pas un théorème *sur* NF : c'est la vérification finie d'une table.
3. **Borne conditionnelle.** Pour `c0-198`, le dépôt relie des machines de 198 états à l'incohérence de C₀ et énonce des bornes de la forme Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity). Une borne conditionnelle se cite **avec sa condition**, jamais comme un fait nu.

Le reste du carnet suit cet ordre : le mécanisme d'abord (§2), les artefacts ensuite (§3), la chaîne de compression comme méthodologie (§3), puis les ponts (§4).

## 1. NF, NFU, C₀ : le décor

La théorie **NF** (New Foundations, Quine, 1937) est une théorie des ensembles à compréhension **stratifiée** : l'ensemble {x | φ(x)} existe lorsque la formule φ respecte une stratification des variables par niveaux. Cette restriction est plus faible que la compréhension non restreinte de la théorie naïve, et plus étrange que celle de ZF : NF admet un ensemble universel et la négation de l'axiome du choix est un théorème. Sa cohérence est un problème **ouvert** depuis près de quatre-vingt-dix ans.

**NFU** est la variante avec ur-éléments (Jensen) ; l'adjonction d'axiomes d'infini et de choix y change la donne. **C₀** est une variante de la même famille, dont le dépôt source calibre des machines sur l'incohérence.

L'idée du **compte à rebours** est alors mécanique. Une machine de Turing à nombre d'états fini peut encoder la recherche de preuves d'une théorie T : elle énumère les dérivations et **s'arrête si et seulement si T dérive une contradiction**. Si l'équivalence « la machine s'arrête ⇔ T est incohérente » est elle-même un théorème — formalisé et audité — alors observer que la machine **ne s'arrête pas** établit Con(T). Autrement dit, le nombre d'états d'une machine devient une unité de mesure de la force d'une théorie : au-delà d'un seuil, la fonction du castor affairé dépasse toute fonction que T prouve totale.

C'est exactement le pont avec [07b](Complexity-07b-LeTourDesGrandsNombres-CSharp.ipynb) : là-bas, le castor affairé mesurait la croissance ; ici, une **borne inférieure** sur sa valeur encode l'indépendance d'une théorie. Et c'est là qu'il faut tenir la distinction : **ce carnet ne parle pas de PA196**. L'indépendance visée ici est celle de **NF**, jamais une fusion avec Peano — le dépôt source écrit lui-même que le paquet à 184 états ne contient pas de nouveau théorème Busy Beaver interne à PA.

## 2. Le mécanisme sur un jouet déclaré

La vraie machine de 184 états encode la logique du premier ordre de NF sur une bande binaire : chaque instruction contribue à l'énumération des preuves. Reproduire cela ici serait une réimplémentation sans licence et sans preuve. À la place, nous construisons **notre propre** machine-jouet sur la logique propositionnelle : un contrôle fini qui énumère les résolutions d'une base de clauses et **s'arrête lorsque la clause vide est dérivée**. Le câblage est le même ; la théorie est minuscule, et c'est précisément ce qui rend le tout exécutable et vérifiable.

Le carnet n'implémente aucun artefact du dépôt source : il implémente l'**idée** du compte à rebours, avec un oracle indépendant (la fermeture par résolution) qui permet de contrôler la machine pas à pas.

In [1]:
// Clauses propositionnelles et oracle de fermeture par résolution
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Security.Cryptography;
using System.Text;

// Le système-jouet : des clauses propositionnelles sur des atomes numérotés.
// +k désigne l'atome x_k, -k sa négation.
public sealed class Clause : IEquatable<Clause>
{
    // Le tableau privé, trié et sans doublon, représente un ensemble immuable.
    private readonly int[] litteraux;

    public Clause(params int[] litteraux)
    {
        this.litteraux = litteraux.Distinct().OrderBy(l => l).ToArray();
    }

    public int Count => litteraux.Length;
    public bool Contient(int litteral) => Array.BinarySearch(litteraux, litteral) >= 0;
    public bool Equals(Clause? autre) => autre is not null && litteraux.SequenceEqual(autre.litteraux);
    public override bool Equals(object? autre) => autre is Clause clause && Equals(clause);

    public override int GetHashCode()
    {
        var hachage = new HashCode();
        foreach (int litteral in litteraux)
            hachage.Add(litteral);
        return hachage.ToHashCode();
    }

    public bool EstTautologie() => litteraux.Any(l => Contient(-l));

    public int[] PivotsAvec(Clause autre) => litteraux.Where(l => autre.Contient(-l)).ToArray();

    public Clause Resoudre(Clause autre, int pivot) => new Clause(
        litteraux.Where(l => l != pivot)
            .Concat(autre.litteraux.Where(l => l != -pivot))
            .ToArray());
}

// Oracle indépendant : chaque passe examine un instantané des clauses connues.
(List<Clause> Clauses, bool Refute) Fermeture(IReadOnlyList<Clause> kb)
{
    var tape = new List<Clause>();
    var vus = new HashSet<Clause>();
    foreach (var clause in kb)
        if (vus.Add(clause))
            tape.Add(clause);

    if (tape.Any(c => c.Count == 0))
        return (tape, true);

    bool change;
    do
    {
        change = false;
        var instantane = tape.ToArray();
        for (int i = 0; i < instantane.Length; i++)
        {
            for (int j = i + 1; j < instantane.Length; j++)
            {
                var c1 = instantane[i];
                var c2 = instantane[j];
                foreach (int pivot in c1.PivotsAvec(c2))
                {
                    var r = c1.Resoudre(c2, pivot);
                    // Une clause vide dérivée donne le verdict sans être ajoutée à la bande.
                    if (r.Count == 0)
                        return (tape, true);
                    if (!r.EstTautologie() && vus.Add(r))
                    {
                        tape.Add(r);
                        change = true;
                    }
                }
            }
        }
    } while (change);
    return (tape, false);
}

void VerifierEgal<T>(string contexte, T calcule, T attendu)
{
    if (!EqualityComparer<T>.Default.Equals(calcule, attendu))
        throw new InvalidOperationException(string.Format(CultureInfo.InvariantCulture,
            "ÉCART — {0} : calculé = {1}, attendu = {2}", contexte, calcule, attendu));
}

var K1 = new List<Clause> { new Clause(1), new Clause(-1, 2), new Clause(-2) };
var K2 = new List<Clause> { new Clause(1, 2), new Clause(-1, 2), new Clause(1, -2), new Clause(-1, -2) };
var K3 = new List<Clause> { new Clause(1, 2), new Clause(-1, 2) };

// Attendus : sorties du carnet Python pour ces trois bases.
var casOracle = new[] { ("K1", K1, 5, true), ("K2", K2, 8, true), ("K3", K3, 3, false) };
foreach (var (nom, kb, effectifAttendu, refuteAttendu) in casOracle)
{
    var (clauses, refute) = Fermeture(kb);
    VerifierEgal($"{nom} — effectif de la fermeture", clauses.Count, effectifAttendu);
    VerifierEgal($"{nom} — réfutation par l'oracle", refute, refuteAttendu);
    Console.WriteLine(FormattableString.Invariant($"{nom} : {clauses.Count} clause(s) dans la fermeture, réfutation = {refute}"));
}

K1 : 5 clause(s) dans la fermeture, réfutation = True


K2 : 8 clause(s) dans la fermeture, réfutation = True


K3 : 3 clause(s) dans la fermeture, réfutation = False


In [2]:
// Un automate explicite rend chaque resolution observable, une transition a la fois.
public enum EtatCompteARebours { ENUM_I, ENUM_J, RESOUT, AVANCE_J, FIN_PASSE, HALTE, BOULCLE }

public sealed class CompteARebours
{
    // La bande contient la base puis ses resolvantes nouvelles ; `vus` evite les doublons.
    private readonly List<Clause> tape = new List<Clause>();
    private readonly HashSet<Clause> vus = new HashSet<Clause>();
    private int i;
    private int j;
    private int[] pivots = Array.Empty<int>();
    private int kp;
    private bool ajoutPasse;

    public IReadOnlyList<Clause> Tape { get; }
    public EtatCompteARebours Etat { get; private set; } = EtatCompteARebours.ENUM_I;
    public int Pas { get; private set; }
    public bool Halte => Etat == EtatCompteARebours.HALTE;

    public CompteARebours(IReadOnlyList<Clause> kb)
    {
        Tape = tape.AsReadOnly();
        foreach (var clause in kb)
            if (vus.Add(clause))
                tape.Add(clause);

        // Une clause vide initiale est deja une refutation : aucun pas n'est necessaire.
        if (tape.Any(c => c.Count == 0))
            Etat = EtatCompteARebours.HALTE;
    }

    // HALTE et BOULCLE sont absorbants : le compteur de pas ne progresse plus.
    public void PasOnce()
    {
        if (Halte || Etat == EtatCompteARebours.BOULCLE)
            return;

        Pas++;
        switch (Etat)
        {
            // ENUM_I choisit la premiere clause de chaque paire ordonnee (i < j).
            case EtatCompteARebours.ENUM_I:
                if (i >= tape.Count)
                    Etat = EtatCompteARebours.FIN_PASSE;
                else
                {
                    j = i + 1;
                    Etat = EtatCompteARebours.ENUM_J;
                }
                break;

            // ENUM_J balaie la seconde clause ; les pivots opposes alimentent RESOUT.
            case EtatCompteARebours.ENUM_J:
                if (j >= tape.Count)
                {
                    i++;
                    Etat = EtatCompteARebours.ENUM_I;
                }
                else
                {
                    pivots = tape[i].PivotsAvec(tape[j]);
                    kp = 0;
                    Etat = EtatCompteARebours.RESOUT;
                }
                break;

            // RESOUT produit les resolvantes une par une, y compris la clause vide.
            case EtatCompteARebours.RESOUT:
                if (kp >= pivots.Length)
                    Etat = EtatCompteARebours.AVANCE_J;
                else
                {
                    int pivot = pivots[kp++];
                    var r = tape[i].Resoudre(tape[j], pivot);
                    // La clause vide donne HALTE ; toute autre clause inedite prolonge la bande.
                    if (r.Count == 0)
                        Etat = EtatCompteARebours.HALTE;
                    else if (!r.EstTautologie() && vus.Add(r))
                    {
                        tape.Add(r);
                        ajoutPasse = true;
                    }
                }
                break;

            // AVANCE_J materialise le pas suivant sans calculer pendant le changement d'indice.
            case EtatCompteARebours.AVANCE_J:
                j++;
                Etat = EtatCompteARebours.ENUM_J;
                break;

            // Une nouvelle passe n'a lieu que si la precedente a ajoute une clause.
            case EtatCompteARebours.FIN_PASSE:
                if (ajoutPasse)
                {
                    ajoutPasse = false;
                    i = 0;
                    Etat = EtatCompteARebours.ENUM_I;
                }
                // Sans nouveauté, le point fixe represente le non-arret du compte a rebours.
                else
                    Etat = EtatCompteARebours.BOULCLE;
                break;
        }
    }
}

// L'observateur borne l'attente ; il ne decide pas a la place de la machine.
CompteARebours Executer(CompteARebours machine, int maxPas = 200_000)
{
    while (!machine.Halte && machine.Etat != EtatCompteARebours.BOULCLE && machine.Pas < maxPas)
        machine.PasOnce();
    return machine;
}

// K1 et K2 se refutent, K3 atteint sa fermeture sans contradiction.
var casMachine = new[]
{
    ("K1", K1, EtatCompteARebours.HALTE, 27, 5),
    ("K2", K2, EtatCompteARebours.HALTE, 87, 8),
    ("K3", K3, EtatCompteARebours.BOULCLE, 36, 3)
};
foreach (var (nom, kb, etatAttendu, pasAttendus, effectifAttendu) in casMachine)
{
    var machine = Executer(new CompteARebours(kb));
    VerifierEgal($"{nom} — état final", machine.Etat, etatAttendu);
    VerifierEgal($"{nom} — nombre de pas", machine.Pas, pasAttendus);
    VerifierEgal($"{nom} — effectif de la bande", machine.Tape.Count, effectifAttendu);
    string verdict = machine.Etat switch
    {
        EtatCompteARebours.HALTE => "HALTE (réfutation dérivée)",
        EtatCompteARebours.BOULCLE => "tourne indéfiniment (fermeture épuisée sans clause vide)",
        _ => "budget épuisé sans verdict"
    };
    Console.WriteLine(FormattableString.Invariant($"{nom} : {verdict} après {machine.Pas} pas -- {machine.Tape.Count} clause(s) sur la bande"));
}

K1 : HALTE (réfutation dérivée) après 27 pas -- 5 clause(s) sur la bande


K2 : HALTE (réfutation dérivée) après 87 pas -- 8 clause(s) sur la bande


K3 : tourne indéfiniment (fermeture épuisée sans clause vide) après 36 pas -- 3 clause(s) sur la bande


In [3]:
var rng = new Random(20261120);

// Chaque tentative tire de un à trois littéraux avec remise, sur les atomes 1 à 3.
List<Clause> KbAleatoire(Random tirage, int nClauses)
{
    var clauses = new List<Clause>();
    for (int tentative = 0; tentative < nClauses; tentative++)
    {
        int taille = tirage.Next(1, 4);
        var litteraux = new int[taille];
        for (int k = 0; k < taille; k++)
        {
            int atome = tirage.Next(1, 4);
            int signe = tirage.Next(2) == 0 ? 1 : -1;
            litteraux[k] = atome * signe;
        }
        var clause = new Clause(litteraux);
        // Comme en Python, les tentatives rejetées ne sont pas remplacées.
        if (!clause.EstTautologie() && !clauses.Contains(clause))
            clauses.Add(clause);
    }
    return clauses;
}

// comptes : accords/desaccords machine-oracle, indecis (ni refute ni boucle prouvee)
const int nombreEssais = 200;
const int budgetCampagne = 50_000;
int accords = 0;
var desaccords = new List<string>();
int indecis = 0;
int pasMaxRefutation = 0;

// campagne : 200 bases aleatoires, la machine doit refuter exactement quand l'oracle deriverait la clause vide
for (int essai = 0; essai < nombreEssais; essai++)
{
    var kb = KbAleatoire(rng, rng.Next(1, 7));
    var (_, refute) = Fermeture(kb);
    var machine = Executer(new CompteARebours(kb), maxPas: budgetCampagne);
    if (refute != machine.Halte)
        desaccords.Add(FormattableString.Invariant($"essai {essai} : oracle={refute}, machine={machine.Halte}"));
    else if (refute && machine.Halte)
    {
        accords++;
        pasMaxRefutation = Math.Max(pasMaxRefutation, machine.Pas);
    }
    else if (!refute && machine.Etat == EtatCompteARebours.BOULCLE)
        accords++;
    else
        indecis++;
}

Console.WriteLine(FormattableString.Invariant($"accords oracle/machine : {accords}/{nombreEssais}"));
Console.WriteLine(FormattableString.Invariant($"désaccords : {desaccords.Count}"));
Console.WriteLine(FormattableString.Invariant($"indécis (budget épuisé avant point fixe) : {indecis}"));
Console.WriteLine(FormattableString.Invariant($"pas maximal observé pour une réfutation (tirages C#) : {pasMaxRefutation}"));
foreach (string ligne in desaccords.Take(5))
    Console.WriteLine("  " + ligne);

// Invariant de la campagne Python : comparaison exacte des trois compteurs, pas des tirages.
VerifierEgal("campagne — nombre de bases classées", accords + desaccords.Count + indecis, 200);
VerifierEgal("campagne — accords oracle/machine", accords, 200);
VerifierEgal("campagne — désaccords", desaccords.Count, 0);
VerifierEgal("campagne — indécis", indecis, 0);

accords oracle/machine : 200/200


désaccords : 0


indécis (budget épuisé avant point fixe) : 0


pas maximal observé pour une réfutation (tirages C#) : 108


### Lecture du résultat

La campagne contrôle la propriété du compte à rebours sur 200 bases tirées avec une graine fixée : quand l'oracle trouve une réfutation, la machine **halte** (et le nombre de pas est mesuré) ; quand l'oracle n'en trouve pas, la machine atteint le point fixe `BOULCLE` sans avoir halté. Aucun désaccord n'est attendu, et la catégorie « indécis » compte les tirages où le budget s'épuise avant le point fixe — une machine réelle, elle, n'a pas de budget : elle tourne.

Deux précautions rendent cette lecture honnête. D'abord, l'accord porte sur un **toy** : trois atomes, des clauses courtes, une fermeture finie — rien de tout cela ne dit quoi que ce soit de NF. Ensuite, la machine-jouet n'est **pas** la machine réelle : elle partage avec elle le câblage (énumération de dérivations, arrêt sur la clause vide, non-arrêt sur fermeture épuisée), pas l'encodage. C'est exactement le geste de la section 0 : distinguer ce qui est prouvé, calculé, et simplement illustré.

**Note de parité — aléa.** `new Random(20261120)` produit des tirages propres à .NET, distincts de ceux du MT19937 de Python. Les bornes restent identiques : 200 bases sur trois atomes, une à six tentatives de clauses par base, sans conserver les tautologies ni les doublons, et un budget de 50 000 pas. Les assertions vérifient le même bilan : 200/200 accords, 0 désaccord et 0 indécis. Le maximum affiché est celui des tirages C# ; il peut varier avec le générateur et n'est pas comparé au maximum Python.

## 3. Les artefacts réels, épinglés (aucun vendoring)

Le dépôt `wlad-svennik/notes` (dernier push lu le 20/11/2026 à 11:19Z) publie, dossier par dossier :

| Dossier | Objet | Statut | Ce que le dépôt publie |
|---|---|---|---|
| `nf-bb210` | Machine à 210 états + développement Lean 4 externe sur NF | table + déclarations + audit enregistré | table, déclarations de théorèmes, audit, instantané 05/11 |
| `nf-bb192` | Machine-compte à rebours à 192 états : s'arrête ssi la NF du premier ordre prouve une contradiction | **théorème vérifié par Lean 4.35.0** sur la table littérale | `COUNTDOWN-PROOF.md`, `NF192Countdown.lean`, preuve informelle PDF, certificats hors ligne |
| `nf-bb185` | Machine à 185 états, même équivalence | **théorème Lean non borné** `NFMachine185.halts_iff_nf_refutation` | audit d'axiomes (`theorem-axioms.txt`) : 3 axiomes standard + 11 vérifications natives ; traces de simulation, volumes dans le dépôt |
| `nf-bb184` | Machine à 184 états, NF pure de Quine | **calcul reproductible** | table `NF184.tm` (SHA-256 `7764cc84…8798`), 7 794 théorèmes signés, 142 608 essais / 88 555 587 pas binaires, 2 809 instructions émettrices, point de contrôle 1,84 Mo, `SHA256SUMS`, scripts de rejeu |
| `c0-198` | Deux machines à 198 états calibrées sur l'incohérence de C₀ | **borne conditionnelle** | rapports de preuve, points de contrôle reproductibles, rapport de réverse-interprétation (Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity)) |

**Ce que « épinglé » veut dire ici** : chaque affirmation renvoie à une révision du dépôt (par exemple `1297cce204` pour le paquet 184, `e0a8130149` et `bfc960b072` pour le 185, `299f86fd7c` et `a006d3abba` pour le 192, `33772f530c` et `9823a07d6b` pour les interprétations, `0a0c8fa599` pour C₀). Le carnet ne recopie ni table ni fichier ; il cite la révision et laisse la vérification au dépôt. La question de licence est ouverte (`wlad-svennik/notes#1`) : tant qu'elle ne reçoit pas de réponse, **zéro vendoring** reste la règle, et chaque greffe se limite à citation + consommation épinglée ou exécution de l'outil cité.

**La chaîne de compression documentée** est `210 → 201 → 192 → 185 → 184`. Le dépôt en expose la méthodologie : table à marqueur variable, testeur `D(x,y)=(J(Kx))y` et formule `((D(x,y)=x)→(D(x,x)=x))→y=x`, réordonnancement des huit rangées, états décodeurs partagés — et **les échecs conservés** (`WORK_NOTES.md` garde un contre-modèle à trois éléments pour `collapsed183`). Le prochain calcul ne fait pas de la physique : il mesure la compression elle-même.

In [4]:
// chaine documentee par le depot : 210 -> 201 -> 192 -> 185 -> 184 etats
var chaine = new (string Nom, int Etats)[]
{
    ("nf-bb210", 210),
    ("étape intermédiaire", 201),
    ("nf-bb192", 192),
    ("nf-bb185", 185),
    ("nf-bb184", 184)
};

// Références : tailles citées et réductions affichées par le carnet Python.
var etatsAttendus = new[] { 210, 201, 192, 185, 184 };
var retraitsAttendus = new[] { 9, 9, 7, 1 };
var pourcentagesAttendus = new[] { "4.29%", "4.48%", "3.65%", "0.54%" };
VerifierEgal("chaîne — nombre de tailles citées", chaine.Length, etatsAttendus.Length);
for (int indice = 0; indice < chaine.Length; indice++)
    VerifierEgal($"chaîne — taille citée ({chaine[indice].Nom})", chaine[indice].Etats, etatsAttendus[indice]);

// Comparaison exacte à la précision affichée par la source : deux décimales.
string PourcentageCompression(int retrait, int tailleInitiale) =>
    (100m * retrait / tailleInitiale).ToString("F2", CultureInfo.InvariantCulture) + "%";

Console.WriteLine("chaîne de compression documentée par le dépôt (états cités) :");
for (int indice = 1; indice < chaine.Length; indice++)
{
    var (precedentNom, precedent) = chaine[indice - 1];
    var (nom, n) = chaine[indice];
    int delta = precedent - n;
    string pourcentage = PourcentageCompression(delta, precedent);
    VerifierEgal($"{precedentNom} -> {nom} — retrait", delta, retraitsAttendus[indice - 1]);
    VerifierEgal($"{precedentNom} -> {nom} — pourcentage", pourcentage, pourcentagesAttendus[indice - 1]);
    Console.WriteLine(FormattableString.Invariant($"  {precedentNom,-20} {precedent,3} -> {nom,-20} {n,3} : -{delta} état(s), {pourcentage} de la taille précédente"));
}

// reduction totale de 210 a 184
int departCompression = chaine[0].Etats;
int arriveeCompression = chaine[chaine.Length - 1].Etats;
int totalCompression = departCompression - arriveeCompression;
string pourcentageTotalCompression = PourcentageCompression(totalCompression, departCompression);
VerifierEgal("compression — retrait total", totalCompression, 26);
VerifierEgal("compression — pourcentage total", pourcentageTotalCompression, "12.38%");
Console.WriteLine(FormattableString.Invariant($"total : -{totalCompression} états ({pourcentageTotalCompression}) de {departCompression} à {arriveeCompression}"));

chaîne de compression documentée par le dépôt (états cités) :


  nf-bb210             210 -> étape intermédiaire  201 : -9 état(s), 4.29% de la taille précédente


  étape intermédiaire  201 -> nf-bb192             192 : -9 état(s), 4.48% de la taille précédente


  nf-bb192             192 -> nf-bb185             185 : -7 état(s), 3.65% de la taille précédente


  nf-bb185             185 -> nf-bb184             184 : -1 état(s), 0.54% de la taille précédente


total : -26 états (12.38%) de 210 à 184


### Lecture de la chaîne

Le calcul ci-dessus montre que la compression est **modeste en pourcentage** : vingt-six états retirés sur deux cent dix, soit un peu plus de douze pour cent. Ce qui compte n'est pas l'ampleur, c'est ce que chaque état retiré **achète** : à chaque étape, la même équivalence d'arrêt doit rester un théorème, sur une table plus petite. Le dépôt documente les échecs intermédiaires, et c'est le point méthodologique du carnet : un contre-modèle à trois éléments pour `collapsed183` est une **information**, pas un déchet — il dit exactement ce qui casse quand on retire un état de trop.

La leçon est transposable : dans toute compression de machine, l'invariant qui doit survivre n'est pas la forme de la table, c'est le **théorème d'équivalence** qui la relie à la théorie. Les états décodeurs partagés sont un levier ; la vérification Lean est le juge.

In [5]:
// Discipline de l'empreinte : on hache ce qu'on possède, jamais ce qu'on cite.
string formuleJouet = "D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x";
string empreinte = Convert.ToHexString(SHA256.HashData(Encoding.UTF8.GetBytes(formuleJouet))).ToLowerInvariant();

// Références exactes tirées de la sortie du carnet Python pour la formule-jouet.
const string formuleAttendue = "D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x";
const string empreinteAttendue = "8c83e4ed6e2eb9e05f81d9f29090af521bfc3af2da20924f9b2a10b409896719";
VerifierEgal("formule-jouet — chaîne exacte", formuleJouet, formuleAttendue);
VerifierEgal("formule-jouet — SHA-256", empreinte, empreinteAttendue);
VerifierEgal("formule-jouet — longueur du SHA-256", empreinte.Length, 64);

Console.WriteLine($"formule-jouet : {formuleJouet}");
Console.WriteLine($"empreinte SHA-256 : {empreinte}");
Console.WriteLine(FormattableString.Invariant($"longueur : {empreinte.Length} caractères hexadécimaux"));

formule-jouet : D(x,y)=(J(Kx))y; ((D(x,y)=x)->(D(x,x)=x))->y=x


empreinte SHA-256 : 8c83e4ed6e2eb9e05f81d9f29090af521bfc3af2da20924f9b2a10b409896719


longueur : 64 caractères hexadécimaux


### Ce que ce carnet ne peut pas faire

Empreinte publiée du paquet 184 : `7764cc84...8798` (citée du dépôt, révision `1297cce204`).
Elle n'est PAS re-vérifiable ici : la table `NF184.tm` n'est pas vendorée (aucune licence).
Citée ne veut pas dire vérifiée -- c'est la règle de la section 0.

- **Il ne vérifie pas la table de 184 états.** Sans vendoring, l'empreinte publiée reste une citation ; le `SHA256SUMS` du dépôt source est l'instrument qui la contrôle, et il s'exécute là-bas.
- **Il ne prouve rien sur NF.** La cohérence de NF reste ouverte ; ce que les machines déplacent, c'est la **forme** de la question : « NF est-elle cohérente ? » devient « cette machine s'arrête-t-elle ? ». Le déplacement est utile parce qu'il rend l'énoncé fini et vérifiable, pas parce qu'il le résout.
- **Il ne fusionne pas les familles.** PA196 (carnet 08d, Greffe A) est une autre théorie, un autre dépôt, un autre statut. Toute prose qui mélange PA et NF rend les deux résultats inutilisables.

## 4. Les sept théories interprétables (pont vers formal_logic)

Le dossier `nf-bb192/interpretations` publie un index théorème par théorème (`SET-THEORY-PROOFS.md`) et une archive `nf-interp.zip` couvrant **sept** théories : NF, FunctionNF, CombNF, M, MAC, M+H et NFU + Infinity. C'est un objet d'une autre nature que les machines : une carte d'**interprétabilité** et d'**équiconsistance**, avec ses fichiers Lean originaux et leurs expositions (inventaire : SET-THEORY-PROOFS.md).

Le pont est direct vers la série `SymbolicAI/Lean` (formal_logic) : l'index cite la formulation exacte de H utilisée, et la question « quelles théories sont équiconsistantes ? » est le pendant sémantique de la question « quelles machines s'arrêtent ? ». L'exploitation de cet index relève d'un jalon distinct (Jalon 3) et exige la lecture préalable de la copie archivée au gisement.

## 5. Exercices

Trois exercices, stubs exécutables sans erreur volontaire (C.1) : le carnet s'exécute de bout en bout même non complétés.

### Exercice 1 — La cible du compte à rebours

Notre machine s'arrête sur la **clause vide**, c'est-à-dire sur une contradiction. Généralisez-la : une machine qui s'arrête sur une clause `cible` donnée, et seulement sur elle. La propriété à vérifier ensuite est la même, décalée d'un cran : `halte <=> cible appartient à la fermeture(kb)`.

In [6]:
// Illustration 1 — La cible du compte à rebours
(bool? Halte, int Pas) HalteSiDerive(IReadOnlyList<Clause> kb, Clause cible, int maxPas = 200_000)
{
    // TODO etudiant : renvoyer (halte, pas) pour une machine qui s'arrête sur `cible`.
    // Vérification attendue : comparer HalteSiDerive(K3, new Clause(2)) à Fermeture(K3).
    return (null, 0);
}

var (halteCible, pasCible) = HalteSiDerive(K3, new Clause(2));
Console.WriteLine(halteCible is null
    ? "Exercice à compléter"
    : FormattableString.Invariant($"K3 dérive {{2}} : halte={halteCible} en {pasCible} pas"));

Exercice à compléter


### Exercice 2 — Rétrécir sans perdre l'équivalence

La chaîne réelle descend de 210 à 184 états. Écrivez le profil de réduction : pour une suite d'entiers, le pourcentage retiré à chaque étape et le total. Puis répondez en une phrase, dans une cellule markdown : quel invariant doit survivre à **chaque** compression, et pourquoi un contre-modèle conservé (le `collapsed183` à trois éléments) vaut-il un résultat ?

In [7]:
// Illustration 2 — Rétrécir sans perdre l'équivalence
#nullable enable
List<double>? ProfilReduction(IReadOnlyList<int> etats)
{
    // TODO etudiant : renvoyer les réductions successives, en pourcentage du précédent.
    return null;
}

var profil = ProfilReduction(new[] { 210, 201, 192, 185, 184 });
Console.WriteLine(profil is null
    ? "Exercice à compléter"
    : "profil : [" + string.Join(", ", profil.Select(p => p.ToString(CultureInfo.InvariantCulture))) + "]");

Exercice à compléter


### Exercice 3 — Classer un résultat avant de le citer

On vous donne trois affirmations. Classez chacune dans l'un des trois statuts de la section 0 : `formalise-audite`, `calcul-reproductible` ou `borne-conditionnelle`.

a) « La machine à 185 états s'arrête si et seulement si NF est réfutable, et l'audit ne liste que les axiomes standard. »
b) « Un simulateur indépendant a exécuté 142 608 essais sur la table à 184 états, sans divergence. »
c) « Si NF est cohérente, alors NFU + Infinity est cohérente. »

In [8]:
// Illustration 3 — Classer un résultat avant de le citer
#nullable enable
string? Classer(string enonce)
{
    // TODO etudiant : renvoyer "formalise-audite", "calcul-reproductible" ou "borne-conditionnelle".
    return null;
}

// Énoncés repris de l'exercice Python, à classer.
var enonces = new[]
{
    "La machine à 185 états s'arrête si et seulement si NF est réfutable (audit d'axiomes publié).",
    "Un simulateur indépendant a exécuté 142 608 essais sur la table à 184 états.",
    "Si NF est cohérente, alors NFU + Infinity est cohérente."
};
foreach (string enonce in enonces)
{
    string? verdictExercice = Classer(enonce);
    Console.WriteLine(verdictExercice is null
        ? "Exercice à compléter"
        : $"{verdictExercice} : {enonce.Substring(0, Math.Min(60, enonce.Length))}...");
}

Exercice à compléter


Exercice à compléter


Exercice à compléter


## 6. Conclusion et ponts

Le compte à rebours est une **traduction** : il transforme une question sur une théorie (« NF est-elle cohérente ? ») en une question sur une machine finie (« s'arrête-t-elle ? »). Trois exigences en découlent, et le carnet les a tenues séparément :

1. **L'équivalence doit être un théorème.** C'est le statut des machines 185 et 192, auditées. Le jouet de la section 2 n'a pas ce statut : il illustre le câblage, il ne prouve rien.
2. **La table doit être reproductible.** C'est le statut du paquet 184 : empreinte, simulateur indépendant, points de contrôle. Citer une empreinte n'est pas la vérifier.
3. **Une borne se cite avec sa condition.** C'est le statut de C₀-198 : Con(NF) ⇒ non-arrêt ⇒ Con(NFU + Infinity), condition comprise.

Les ponts naturels sont la série `SymbolicAI/Lean` (l'exploitation de l'index d'interprétabilité des sept théories relève du Jalon 3) et le [carnet 07b](Complexity-07b-LeTourDesGrandsNombres-CSharp.ipynb) (le castor affairé comme fonction de croissance, dont ces machines calculent des valeurs inférieures). Le Jalon 2 prévoit un lac consommateur `nf_bb_lean` épinglé sur NF185, sous réserve qu'une fenêtre de build soit disponible.

## 7. Références

**Source primaire — dépôt `wlad-svennik/notes`** (lu firsthand le 20/11/2026, dernier push 11:19Z ; révisions épinglées citées en section 3). Aucune licence détectée dans le dépôt ; demande de clarification ouverte : `wlad-svennik/notes#1`. Aucun contenu recopié.

**Copies archivées au gisement** (bibliography-hygiene, identités vérifiées page 1), rayon `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\` :

- `2026 - Svennik (wlad) - 192-state PA bound for non-regularity (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - 192-state countdown machine and NF (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - tangled type theory and Con NF (github wlad-svennik-notes, sans licence).pdf`
- `2026 - Svennik (wlad) - 201-state Zermelo (github wlad-svennik-notes, sans licence).pdf`

**Références de fond, non consommées dans ce carnet** : Quine, *New Foundations for Mathematical Logic* (1937) — compréhension stratifiée ; Jensen, résultats de cohérence de NFU ; le carnet [Complexity-07b](Complexity-07b-LeTourDesGrandsNombres-CSharp.ipynb) pour le castor affairé et Tibor Radó (1962) ; [Complexity-08d](Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb) pour PA196, qui n'est **pas** fusionné avec ce carnet.

### Attestation de parité

Les assertions interrompent l'exécution si une comparaison échoue. Une exécution intégrale sans erreur atteste le bilan suivant :

| Section | Comparaison exécutée | Écart si les assertions passent |
|---|---|---|
| §2 — Oracle | Effectifs de fermeture et verdicts de K1, K2 et K3 | 0 clause d'écart et aucun verdict divergent |
| §2 — Automate | États finaux, nombres de pas et tailles des bandes pour les trois bases | 0 pas et 0 clause d'écart ; aucun état final divergent |
| §2 — Campagne | Bilan machine–oracle : 200/200 accords, 0 désaccord et 0 indécis | 0 sur chacun des compteurs ; le maximum de pas est propre aux tirages C# |
| §3 — Compression | Tailles citées, retraits recalculés, pourcentages à deux décimales et total | 0 état et 0.00 point de pourcentage d'écart à la précision affichée |
| §3 — Empreinte | Texte de la formule-jouet, SHA-256 UTF-8 et longueur | Aucune différence de texte ou de condensat ; 0 caractère d'écart sur la longueur |

L'empreinte publiée de `NF184.tm` reste une citation et ne fait pas partie des comparaisons.

> Ce carnet est le jumeau C# de la [version Python](Complexity-08e-NF-MachinesCompteARebours-Python.ipynb).